# 01 · System B — few-shot hierarchical cascade

A **training-free** pipeline: an open-weight vision-language model is prompted
with a definition-rich system prompt and a handful of exemplars, and answers
the three subtasks in a cascade. No fine-tuning, no gradient, one code path for
memes and videos.

```
                    ┌──────────────────────────────────────────┐
  media + text ───▶ │  x.1  sexist?                YES / NO     │
                    └───────────────┬──────────────────────────┘
                          YES       │        NO
                    ┌───────────────▼──────┐   └──▶ x.2 = NO, x.3 = [NO]
                    │ x.2 DIRECT/JUDGEMENT.│        (propagated, not asked)
                    │ x.3 five categories  │
                    └──────────────────────┘
```

**What this notebook is for.** Reading the pipeline and inspecting one run:
the pools it selected, the prompt it will send, the token budget, the metrics
and the errors. The pipeline itself lives in `exist2026.fewshot`, so a full run
belongs on the command line, where it survives a disconnect:

```bash
python scripts/run_fewshot.py --config fewshot_memes  --stage sanity
python scripts/run_fewshot.py --config fewshot_videos --stage submission
```

Every step below is idempotent and checkpointed: re-running after an
interruption skips what has already been predicted.

In [ ]:
# Makes the notebook work from a clone (no install) and on Colab alike.
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

# Point this at the directory holding the EXIST 2026 corpus. The configs read
# it from here, so nothing below contains a hard-coded path.
import os
os.environ.setdefault("EXIST2026_ROOT", str(Path.home() / "EXIST_2026"))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1 · Configuration

One switch. `MODALITY` picks the corpus, the official thresholds, the prompts
and the pool design; `BACKEND` picks the model; `STAGE` picks what to score.

- `sanity` → a stratified TRAIN subset, scored with PyEvALL (hard-hard). Run
  this before spending GPU hours.
- `submission` → the whole official TEST split, packaged for upload.

In [ ]:
MODALITY = "memes"       # "memes" | "videos"
BACKEND  = "gemma4"      # "gemma4" | "qwen35"
STAGE    = "sanity"      # "sanity" | "submission"

from exist2026.config import load_fewshot_config

config = load_fewshot_config(f"fewshot_{MODALITY}", stage=STAGE, backend=BACKEND)
config.dirs.create()

print(f"modality  : {config.modality.value}")
print(f"backend   : {config.backend.name} ({config.backend.model_id})")
print(f"stage     : {config.stage}  ->  run mode {config.run_mode}")
print(f"subtasks  : {config.subtask_keys}")
print(f"outputs   : {config.dirs.root}")

## 2 · Data

TRAIN is always loaded, even for a submission run: the few-shot pools are built
from it, and building them anywhere else would leak test instances into the
prompts.

In [ ]:
from exist2026.datasets import describe, load_corpus

needs_test = config.run_mode == "test"
corpus = load_corpus(
    config.modality,
    config.thresholds,
    train_json=config.data.train_json(),
    train_media_dir=config.data.train_media_dir(),
    test_json=config.data.test_json() if needs_test else None,
    test_media_dir=config.data.test_media_dir() if needs_test else None,
)
describe(corpus)

## 3 · Few-shot pools

Selected from TRAIN, cached on disk, and **excluded from evaluation** — an
exemplar the model has been shown is not a test of anything.

Three properties are enforced in `exist2026.fewshot.pools`:

- **Ranked by annotator agreement**, so the exemplars are the clearest cases.
  Video pools mix in medium-agreement instances on purpose: a prompt made only
  of unanimous cases never shows the model where the boundary is.
- **Hierarchy-consistent**: the x.2 and x.3 pools contain no `NO` exemplar,
  because those subtasks only ever receive instances the cascade already gated.
- **Deterministic**: ties break on id, so the same corpus always yields the same
  prompt. Passing a seed instead adds a random tie-break, which is how pool
  variance is measured.

In [ ]:
from exist2026.fewshot import pools as pool_module

pools = pool_module.load_or_build_pools(corpus, config.pools, config.dirs.pools)
blocked_ids = pool_module.pool_ids(pools)

print(f"{len(blocked_ids)} instances held out of evaluation\n")
pool_module.summarize(pools)

## 4 · Prompts

The prompt text is **not** in this notebook or in the package: it lives in
`prompts/<modality>/`, one file per subtask, and is read at run time. The
published prompts and the bytes actually sent are therefore the same thing.

In [ ]:
from exist2026.fewshot.prompts import builders_for

builders = builders_for(
    config.modality,
    use_rationale=config.use_rationale,
    fps=config.video_fps if MODALITY == "videos" else None,
    max_pixels=config.video_max_pixels if MODALITY == "videos" else None,
)

key = config.subtask_keys[0]
print(builders[key].system[:1200], "\n[...]")

In [ ]:
# The full message list for one instance: system, then the exemplars as
# user/assistant pairs, then the query with no answer.
from exist2026.datasets import exclude

split = corpus.test if config.run_mode == "test" else corpus.train
query = exclude(split, blocked_ids).iloc[0]

messages = builders[key].build(
    query["media_path"], query["text"], pools[key], corpus.media_of, corpus.text_of
)
for message in messages[1:]:
    content = message["content"]
    if isinstance(content, str):
        print(f"[{message['role']:9s}] {content}")
    else:
        media = content[0]
        print(f"[{message['role']:9s}] <{media['type']}> {content[1]['text'][:110]}...")

## 5 · Backend

Qwen-VL and Gemma-4 differ only in how inputs are prepared; everything above
and below this cell is identical for both. Loading falls back across candidate
model classes because the concrete class name for a recent VLM moves between
`transformers` releases.

Decoding is greedy (`do_sample=False`) so a run is reproducible.

In [ ]:
from exist2026.fewshot.backends import load_backend
from exist2026.fewshot.parsing import ParserPolicy, ResponseParser
from exist2026.fewshot.pipeline import check_token_budget

backend = load_backend(config.backend, config.generation)

parser = ResponseParser(
    ParserPolicy.for_modality(
        config.modality, config.categorization_fallback or corpus.most_frequent_category()
    ),
    log_path=config.dirs.logs / "parse_fallbacks.log",
)

# Video prompts carry several sampled frames per exemplar: check they fit
# before starting a run that takes hours to discover otherwise.
check_token_budget(backend, builders, pools, corpus, config, blocked_ids)

## 6 · Run the cascade

x.1 over everything, then x.2 / x.3 over the instances x.1 called sexist. The
rest inherit `NO` without a generation, which halves the work and keeps the
three subtasks consistent with one another.

Checkpoints are written every `checkpoint_every` instances; a failed generation
costs one instance, not the run.

In [ ]:
from exist2026.fewshot.pipeline import CheckpointStore, run_cascade

store = CheckpointStore(config.dirs.checkpoints, config.checkpoint_suffix)
result = run_cascade(
    backend=backend,
    builders=builders,
    pools=pools,
    parser=parser,
    corpus=corpus,
    config=config,
    store=store,
    blocked_ids=blocked_ids,
)

## 7 · How much of the output was actually parsed

A silent fallback rate is indistinguishable from a working prompt in the
metrics alone, so it is reported separately. Parser misses point at the prompt
or the token budget; exceptions point at VRAM or missing media. Above ~5%,
fix that before reading any score.

In [ ]:
parser.report(config.subtask_keys)

## 8 · Score, or package

In `sanity` we score hard-hard against the TRAIN gold and look at the errors.
In `submission` there is no gold, so we check the prediction distribution for
anything implausible and write the upload.

In [ ]:
from exist2026.submission import build_records, write_predictions

for subtask_key, predictions in result.predictions.items():
    path = config.dirs.predictions / f"pred_{subtask_key}_hard_{config.checkpoint_suffix}.json"
    write_predictions(path, predictions, subtask_key)
    print(f"[{subtask_key}] {len(predictions):,} predictions -> {path.name}")

In [ ]:
from exist2026.evaluation import HARD_METRICS, evaluate, hard_gold
from exist2026.evaluation.reports import compare, confusion, prediction_distribution

if config.run_mode == "train_eval":
    scores = []
    for subtask_key, predictions in result.predictions.items():
        records = build_records(predictions)
        gold = hard_gold(corpus, [r["id"] for r in records], subtask_key)
        scores.append({
            "subtask": subtask_key,
            **evaluate(records, gold, subtask_key, HARD_METRICS, work_dir=config.dirs.pyevall_work),
        })
    display(pd.DataFrame(scores))
else:
    for subtask_key, predictions in result.predictions.items():
        print(f"\n[{subtask_key}]")
        display(prediction_distribution(predictions))

### Error analysis (sanity runs)

x.2 and x.3 are compared **only on instances whose gold is sexist**. Where x.1
said NO the downstream label was propagated by the cascade rather than chosen
by the model, so scoring those rows would measure the hierarchy and inflate
accuracy with free NOs.

In [ ]:
if config.run_mode == "train_eval":
    from exist2026.evaluation.reports import save_errors

    for subtask_key in config.subtask_keys:
        comparison = compare(result.predictions[subtask_key], corpus, subtask_key)
        print(f"\n[{subtask_key}] {comparison.metrics}")
        if comparison.note:
            print(f"  note: {comparison.note}")
        saved = save_errors(comparison, corpus, config.dirs.metrics / "errors")
        if saved:
            print(f"  errors -> {saved}")

    identification = config.subtask_keys[0]
    display(confusion(compare(result.predictions[identification], corpus, identification), ["NO", "YES"]))

### Baseline floor

Always compare against the trivial system. A pipeline that does not beat "the
majority training class, propagated through the hierarchy" is not working,
whatever its absolute numbers look like.

In [ ]:
if config.run_mode == "train_eval":
    from exist2026.fewshot.pipeline import majority_baseline

    baseline_scores = []
    for subtask_key, predictions in majority_baseline(config, corpus, blocked_ids).items():
        records = build_records(predictions)
        gold = hard_gold(corpus, [r["id"] for r in records], subtask_key)
        baseline_scores.append({
            "subtask": subtask_key,
            **evaluate(records, gold, subtask_key, HARD_METRICS, work_dir=config.dirs.pyevall_work),
        })
    display(pd.DataFrame(baseline_scores))

## 9 · Submission

The organizers accept a narrow format, so `package()` validates every record
before writing: the mandated `test_case`, the label space of each subtask, `NO`
being exclusive in x.3, and full coverage of the split.

In [ ]:
if config.run_mode == "test":
    from exist2026.submission import check_coverage, package

    expected = set(corpus.test["id_EXIST"])
    for subtask_key, predictions in result.predictions.items():
        check_coverage(predictions, expected, subtask_key)

    base = package(
        result.predictions,
        modality=config.modality,
        output_dir=config.dirs.submission,
        team_name=config.team_name,
        run_id=config.run_id,
    )
    print(f"ready to upload: {base}.zip")